# DATA EXTRACTION VIA API 

In [ ]:
import pandas as pd
import yfinance as yf
import wrds
import pandas as pd
import os
import re
import json
from pathlib import Path
from datetime import datetime
import wrds
import psycopg2
from psycopg2.extras import json as psycop_json

In [ ]:
# CONFIQ
WRDS_USERNAME = "tnbtran2023"
INPUT_CSV = "TechCompanyByMarketCap_withCIK.csv"
FILING_TYPES = ["10-K", "10-Q"]
START_DATE = "2021-01-01"
END_DATE = "2025-12-31"

# Output directories
OUTPUT_DIR = "../MDA"
RAW_FILINGS_DIR = "../Filings"

# ========================= CONNECTION SETUP ==================================
def get_wrds_connection(username):
    """
    Establish connection to WRDS PostgreSQL database.
    Follows the exact same connection pattern from the reference notebook.
    You will be prompted for your WRDS password.
    """
    conn = psycopg2.connect(
        host="wrds-pgdata.wharton.upenn.edu",
        database="wrds",
        user=username,
        port=9737
    )
    conn.autocommit = True
    return conn


In [ ]:

# STEP 1: READ CSV AND EXTRACT CIK LIST
def load_companies_from_csv(csv_path):
    """
    csv already includes CIK of companies
    """
    df = pd.read_csv(csv_path, dtype=str)
    df.columns = [c.strip() for c in df.columns]

    # Find the CIK column (case-insensitive match)
    cik_col = None
    for col in df.columns:
        if "cik" in col.lower():
            cik_col = col
            break
    if cik_col is None:
        raise ValueError(
            f"No CIK column found in CSV. Columns are: {list(df.columns)}\n"
            f"Please ensure your CSV has a column with 'CIK' in its name."
        )

    # Try to find a company name column (optional, for labelling)
    name_col = None
    for col in df.columns:
        if col.lower() != cik_col.lower() and any(
            kw in col.lower() for kw in ["name", "company", "firm", "ticker"]
        ):
            name_col = col
            break

    companies = []
    for _, row in df.iterrows():
        raw_cik = str(row[cik_col]).strip()
        if not raw_cik or raw_cik.lower() == "nan":
            continue

        # Zero-pad CIK to 10 digits (standard SEC format)
        cik = raw_cik.lstrip("0")
        cik_padded = cik.zfill(10)

        name = ""
        if name_col and pd.notna(row.get(name_col)):
            name = str(row[name_col]).strip()

        companies.append({"cik": cik_padded, "cik_raw": raw_cik, "name": name})

    return companies[68:101]



In [ ]:

# =============================================================================
# ================ STEP 2: RETRIEVE FILINGS FROM WRDS ========================
# =============================================================================

def get_filings(conn, cik_list, filing_types, start_date, end_date):
    """
    Retrieve filings from WRDS wrds_sec_search tables.
    Mirrors the reference notebook query pattern exactly.
    
    Parameters:
        conn: psycopg2 connection
        cik_list: list of CIK strings (e.g., ['0001418091'])
        filing_types: list of form types (e.g., ['10-K', '10-Q'])
        start_date: 'YYYY-MM-DD' string
        end_date: 'YYYY-MM-DD' string
    
    Returns list of tuples: (accession, form, filing_date, report_date, 
                              acceptance, registrants, filing_text)
    """
    # Build the SQL IN clauses exactly as the notebook does
    cik_in = ", ".join([f"'{c}'" for c in cik_list])
    form_in = ", ".join([f"'{f}'" for f in filing_types])

    with conn.cursor() as cursor:
        sql_query = f"""
            SELECT
                filing_view.accession,
                filing_view.form,
                filing_view.filing_date,
                filing_view.report_date,
                filing_view.acceptance,
                filing_view.registrants,
                filing_view.filing
            FROM
                wrds_sec_search.filing_view
            JOIN
                wrds_sec_search.registrant 
                ON registrant.accession = filing_view.accession
            WHERE
                form IN ({form_in})
                AND wrds_sec_search.registrant.cik IN ({cik_in})
                AND filing_view.filing_date >= '{start_date}'
                AND filing_view.filing_date <= '{end_date}'
            ORDER BY
                filing_view.filing_date ASC
        """
        cursor.execute(sql_query, [])
        results = cursor.fetchall()
    return results


In [ ]:
# =============================================================================
# ================ STEP 3: EXTRACT MD&A SECTION ==============================
# =============================================================================
def extract_mda(text):
    text = re.sub(r'\s+', ' ', text)

    mda_start_patterns = re.compile(
        '|'.join([
            # Handles "ITEM 2.MANAGEMENTS" (no space, no apostrophe) — the actual body header
            r'item\s*2[\.\:\s\-\u2014]*managements?\s+discussion\s+and\s+analysis',
            # Handles standard "Item 2. Management's Discussion..." with optional apostrophe/space
            r'item\s*2[\.\:\s\-\u2014]+management[\'\u2019]?s?\s+discussion\s+and\s+analysis',
            # Item 7 variant for annual reports
            r'item\s*7[\.\:\s\-\u2014]+management[\'\u2019]?s?\s+discussion\s+and\s+analysis',
            # Standalone header fallback
            r'management[\'\u2019]?s?\s+discussion\s+and\s+analysis\s+of\s+financial\s+condition',
        ]),
        re.IGNORECASE
    )

    mda_end_patterns = re.compile(
        '|'.join([
            r'item\s*7a[\.\:\s\-\u2014]*quantitative\s+and\s+qualitative',
            r'item\s*8[\.\:\s\-\u2014]+financial\s+statements',
            # Handles "ITEM 3.QUANTITATIVE" (no space after period)
            r'item\s*3[\.\:\s\-\u2014]*quantitative\s+and\s+qualitative',
            r'item\s*4[\.\:\s\-\u2014]*controls\s+and\s+procedures',
        ]),
        re.IGNORECASE
    )

    start_matches = [m.start() for m in mda_start_patterns.finditer(text)]
    end_matches   = [m.start() for m in mda_end_patterns.finditer(text)]

    if not start_matches or not end_matches:
        raise ValueError("Could not find MD&A start or end marker")

    MIN_MDA_LENGTH = 5000  # real MD&A sections are tens of thousands of chars

    for start in start_matches:
        end_candidates = [x for x in end_matches if x > start]
        if not end_candidates:
            continue
        end = end_candidates[0]
        if end - start > MIN_MDA_LENGTH:
            return text[start:end]

    raise ValueError("Valid MD&A section not found (all candidates too short)")



In [ ]:
# =============================================================================
# ================ STEP 4: CLEAN EXTRACTED TEXT ===============================
# =============================================================================

def clean_mda_text(mda_text):
    """
    Clean the extracted MD&A text by removing HTML tags, 
    excessive whitespace, and other artifacts.
    """
    if not mda_text:
        return None

    # Remove HTML/XML tags
    cleaned = re.sub(r'<[^>]+>', ' ', mda_text)

    # Remove XBRL tags
    cleaned = re.sub(r'\{[^}]+\}', '', cleaned)

    # Collapse multiple whitespace/newlines
    cleaned = re.sub(r'\s+', ' ', cleaned)

    # Collapse multiple periods/dashes
    cleaned = re.sub(r'\.{3,}', '...', cleaned)
    cleaned = re.sub(r'-{3,}', '---', cleaned)

    return cleaned.strip()

In [ ]:

# ========================================================================
# ================ EXECUTION =============================================
# ========================================================================

def main():

    # # ---- Load companies from CSV ----
    print("=" * 70)
    print("WRDS SEC MD&A Extractor")
    print("=" * 70)
    print(f"Input CSV:      {INPUT_CSV}")
    print(f"Filing Types:   {FILING_TYPES}")
    print(f"Date Range:     {START_DATE} to {END_DATE}")
    print(f"Raw Filings:    {RAW_FILINGS_DIR}")
    print(f"MD&A Output:    {OUTPUT_DIR}")
    print("=" * 70)

    companies = load_companies_from_csv(INPUT_CSV)
    print(f"\nLoaded {len(companies)} companies from CSV:")
    for c in companies:
        label = c['name'] if c['name'] else "(no name column)"
        print(f"  CIK: {c['cik']}  |  {label}")

    # ---- Connect to WRDS ----
    print("\nConnecting to WRDS...")
    conn = get_wrds_connection(WRDS_USERNAME)
    print("Connected successfully.\n")

    # ---- Process each company ----
    summary_rows = []

    for comp in companies:
        # cik = comp["cik"]
        # label = comp["name"] if comp["name"] else f"CIK_{cik}"

        # print(f"{'─' * 60}")
        # print(f"  {label}  (CIK: {cik})")
        # print(f"{'─' * 60}")

        total_companies = len(companies)
        for idx, comp in enumerate(companies, 1):
            cik = comp["cik"]
            label = comp["name"] if comp["name"] else f"CIK_{cik}"

            print(f"  [{idx}/{total_companies}] {label}  (CIK: {cik})")

        # Retrieve filings
        print(f"  Fetching {FILING_TYPES} filings ({START_DATE} to {END_DATE})...")

        # Try both padded and unpadded CIK to maximise match
        cik_variants = list(set([cik, cik.lstrip("0"), cik.zfill(10)]))
        filings = get_filings(conn, cik_variants, FILING_TYPES, START_DATE, END_DATE)
        print(f"  Found {len(filings)} filings.")

        if not filings:
            print(f"  WARNING: No filings found. Skipping.\n")
            summary_rows.append({
                "cik": comp["cik_raw"],
                "company_name": label,
                "filings_found": 0,
                "filings_saved": 0,
                "mda_extracted": 0,
            })
            continue

        # ---- Save raw filings to disk first ----
        safe_label = re.sub(r'[^\w\-]', '_', label.strip())
        company_raw_dir = os.path.join(RAW_FILINGS_DIR, safe_label)

        saved_count = 0
        for filing in filings:
            accession, form, filing_date, report_date, acceptance, registrants, filing_text = filing
            if not filing_text:
                continue
            raw_filename = f"{safe_label}_{form}_{filing_date}.txt"
            raw_filepath = os.path.join(company_raw_dir, raw_filename)
            with open(raw_filepath, "w", encoding="utf-8") as f:
                f.write(filing_text)
            saved_count += 1

        print(f"  Saved {saved_count}/{len(filings)} raw filings to {company_raw_dir}/")

        # ---- Extract MD&A from each filing ----
        mda_count = 0
        for filing in filings:
            accession, form, filing_date, report_date, acceptance, registrants, filing_text = filing

            mda_raw = extract_mda(filing_text)
            if mda_raw is None:
                print(f"    {form} | {filing_date} | {accession} | MD&A: NOT FOUND")
                continue

            mda_cleaned = clean_mda_text(mda_raw)
            if mda_cleaned is None or len(mda_cleaned) < 200:
                print(f"    {form} | {filing_date} | {accession} | MD&A: TOO SHORT")
                continue

            mda_count += 1
            print(f"    {form} | {filing_date} | {accession} | MD&A: {len(mda_cleaned):,} chars")

            # Save MD&A to file
            filename = f"{safe_label}_{form}_{filing_date}_MDA.txt"
            filepath = os.path.join(OUTPUT_DIR, filename)

            with open(filepath, "w", encoding="utf-8") as f:
                f.write(mda_cleaned)

        print(f"  Result: {mda_count}/{len(filings)} MD&A sections extracted.\n")

        summary_rows.append({
            "cik": comp["cik_raw"],
            "company_name": label,
            "filings_found": len(filings),
            "filings_saved": saved_count,
            "mda_extracted": mda_count,
        })

    # ---- Save summary ----
    summary_path = os.path.join(OUTPUT_DIR, "_extraction_summary.csv")
    df_summary = pd.DataFrame(summary_rows)
    df_summary.to_csv(summary_path, index=False)

    conn.close()

    print(f"{'=' * 70}")
    print("EXTRACTION COMPLETE")
    print(f"{'=' * 70}")
    print(df_summary.to_string(index=False))
    print(f"\nRaw filings:     {os.path.abspath(RAW_FILINGS_DIR)}/")
    print(f"MD&A files:      {os.path.abspath(OUTPUT_DIR)}/")
    print(f"Summary CSV:     {summary_path}")
    print(f"{'=' * 70}")


if __name__ == "__main__":
    main()


In [ ]:
with open("Filings/Autodesk/Autodesk_10-Q_2025-11-26.txt", "r", encoding="utf-8") as f:
    sec_text = f.read()

def extract_mda(text):
    text = re.sub(r'\s+', ' ', text)

    mda_start_patterns = re.compile(
        '|'.join([
            # Handles "ITEM 2.MANAGEMENTS" (no space, no apostrophe) — the actual body header
            r'item\s*2[\.\:\s\-\u2014]*managements?\s+discussion\s+and\s+analysis',
            # Handles standard "Item 2. Management's Discussion..." with optional apostrophe/space
            r'item\s*2[\.\:\s\-\u2014]+management[\'\u2019]?s?\s+discussion\s+and\s+analysis',
            # Item 7 variant for annual reports
            r'item\s*7[\.\:\s\-\u2014]+management[\'\u2019]?s?\s+discussion\s+and\s+analysis',
            # Standalone header fallback
            r'management[\'\u2019]?s?\s+discussion\s+and\s+analysis\s+of\s+financial\s+condition',
        ]),
        re.IGNORECASE
    )

    mda_end_patterns = re.compile(
        '|'.join([
            r'item\s*7a[\.\:\s\-\u2014]*quantitative\s+and\s+qualitative',
            r'item\s*8[\.\:\s\-\u2014]+financial\s+statements',
            # Handles "ITEM 3.QUANTITATIVE" (no space after period)
            r'item\s*3[\.\:\s\-\u2014]*quantitative\s+and\s+qualitative',
            r'item\s*4[\.\:\s\-\u2014]*controls\s+and\s+procedures',
        ]),
        re.IGNORECASE
    )

    start_matches = [m.start() for m in mda_start_patterns.finditer(text)]
    end_matches   = [m.start() for m in mda_end_patterns.finditer(text)]

    if not start_matches or not end_matches:
        raise ValueError("Could not find MD&A start or end marker")

    MIN_MDA_LENGTH = 5000  # real MD&A sections are tens of thousands of chars

    for start in start_matches:
        end_candidates = [x for x in end_matches if x > start]
        if not end_candidates:
            continue
        end = end_candidates[0]
        if end - start > MIN_MDA_LENGTH:
            return text[start:end]

    raise ValueError("Valid MD&A section not found (all candidates too short)")

mda_raw = extract_mda(sec_text)
print(mda_raw)